# Feature Engineering + Feature Selection (Combined, Generalized)

Based on your original workflow: missing-value indicator flags, target-guided categorical
encoding, rare-label grouping, skew correction, MinMax scaling, and Lasso-based feature
selection — generalized so it isn't tied to House Prices column names, and with the scaler
fit only on train and correctly reapplied to test (no re-fitting, no data leakage).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

%matplotlib inline

from scipy import stats

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split

pd.pandas.set_option("display.max_columns", None)


## Config

Adjust these for your dataset.

In [2]:
# ==========================================================
# CONFIG
# ==========================================================

INPUT_FILE = "train.csv"

# Default 80/20 split
TEST_SIZE = 0.20
RANDOM_STATE = 42

TARGET_COL = "SalePrice"

RARE_LABEL_THRESHOLD = 0.01
SKEW_THRESHOLD = 0.75
LASSO_ALPHA = 0.0005

# Optional: temporal columns
TEMPORAL_COLS = []
TEMPORAL_REFERENCE_COL = None

# Create outputs automatically in the same working folder.
OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

X_TRAIN_OUTPUT = OUTPUT_DIR / f"X_train_{RUN_TIMESTAMP}.csv"
X_TEST_OUTPUT = OUTPUT_DIR / f"X_test_{RUN_TIMESTAMP}.csv"

print(f"Output folder: {OUTPUT_DIR.resolve()}")
print(f"Run timestamp: {RUN_TIMESTAMP}")


Output folder: C:\Koustav\Programming\Machine Learning\Projects\Machine_learning_automation\Data_Preprocessing\outputs
Run timestamp: 20260811_193942


## Load training data

In [3]:
# Load the complete dataset first.
full_dataset = pd.read_csv(INPUT_FILE)

print(
    f"Loaded {full_dataset.shape[0]} rows, "
    f"{full_dataset.shape[1]} columns"
)

if TARGET_COL not in full_dataset.columns:
    raise ValueError(
        f"Target column '{TARGET_COL}' was not found in the dataset."
    )

# Split BEFORE fitting any preprocessing step.
# Stratification is used for categorical targets when possible.
target_is_categorical = (
    full_dataset[TARGET_COL].dtype == "O"
    or str(full_dataset[TARGET_COL].dtype).startswith("category")
)

stratify_arg = (
    full_dataset[TARGET_COL]
    if target_is_categorical
    else None
)

try:
    train_dataset, test_dataset = train_test_split(
        full_dataset,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=stratify_arg
    )
except ValueError:
    # If stratification is impossible because a class is too small,
    # fall back to a normal random split.
    print(
        "Warning: stratified split was not possible. "
        "Falling back to a regular random split."
    )

    train_dataset, test_dataset = train_test_split(
        full_dataset,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

train_dataset = train_dataset.reset_index(drop=True)
test_dataset = test_dataset.reset_index(drop=True)

print(f"Training rows: {len(train_dataset)}")
print(f"Test rows: {len(test_dataset)}")
print(f"Test size: {TEST_SIZE}")

# All preprocessing below is fitted using train_dataset only.
dataset = train_dataset.copy()
test = test_dataset.copy()

dataset.head()


Loaded 1460 rows, 81 columns
Training rows: 1168
Test rows: 292
Test size: 0.2


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,255,20,RL,70.0,8400,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,1Fam,1Story,5,6,1957,1957,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,Gd,CBlock,TA,TA,No,Rec,922,Unf,0,392,1314,GasA,TA,Y,SBrkr,1314,0,0,1314,1,0,1,0,3,1,TA,5,Typ,0,NaN,Attchd,1957.0,RFn,1,294,TA,TA,Y,250,0,0,0,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal,145000
1,1067,60,RL,59.0,7837,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,6,7,1993,1994,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,Gd,TA,PConc,Gd,TA,No,Unf,0,Unf,0,799,799,GasA,Gd,Y,SBrkr,799,772,0,1571,0,0,2,1,3,1,TA,7,Typ,1,TA,Attchd,1993.0,RFn,2,380,TA,TA,Y,0,40,0,0,0,0,NaN,NaN,NaN,0,5,2009,WD,Normal,178000
2,639,30,RL,67.0,8777,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Edwards,Feedr,Norm,1Fam,1Story,5,7,1910,1950,Gable,CompShg,MetalSd,Wd Sdng,NaN,0.0,TA,TA,CBlock,Fa,TA,No,Unf,0,Unf,0,796,796,GasA,Gd,Y,FuseA,796,0,0,796,0,0,1,0,2,1,TA,4,Typ,0,NaN,NaN,NaN,NaN,0,0,NaN,NaN,P,328,0,164,0,0,0,NaN,MnPrv,NaN,0,5,2008,WD,Normal,85000
3,800,50,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,Corner,Gtl,SWISU,Feedr,Norm,1Fam,1.5Fin,5,7,1937,1950,Gable,CompShg,Wd Sdng,Wd Sdng,BrkFace,252.0,TA,TA,BrkTil,Gd,TA,No,ALQ,569,Unf,0,162,731,GasA,Ex,Y,SBrkr,981,787,0,1768,1,0,1,1,3,1,Gd,7,Typ,2,TA,Detchd,1939.0,Unf,1,240,TA,TA,Y,0,0,264,0,0,0,NaN,MnPrv,NaN,0,6,2007,WD,Normal,175000
4,381,50,RL,50.0,5000,Pave,Pave,Reg,Lvl,AllPub,Inside,Gtl,SWISU,Norm,Norm,1Fam,1.5Fin,5,6,1924,1950,Gable,CompShg,BrkFace,Wd Sdng,NaN,0.0,TA,TA,BrkTil,TA,TA,No,LwQ,218,Unf,0,808,1026,GasA,TA,Y,SBrkr,1026,665,0,1691,0,0,2,0,3,1,Gd,6,Typ,1,Gd,Detchd,1924.0,Unf,1,308,TA,TA,Y,0,0,242,0,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal,127000


## Drop identifier-like columns (universal — no hardcoded column name)

Any column where every value is unique is almost certainly an ID column with no predictive
value, regardless of its name. We keep it aside (not just drop it) so it can be reattached
to test predictions later for submission purposes.

In [4]:
id_like_cols = [
    col for col in dataset.columns
    if col != TARGET_COL
    and dataset[col].nunique(dropna=False) == len(dataset)
]

print(
    f"Identifier-like columns detected: {id_like_cols}"
)

ID_COL = id_like_cols[0] if id_like_cols else None

if id_like_cols:

    train_ids = dataset[id_like_cols].copy()
    test_ids = test[id_like_cols].copy()

    dataset = dataset.drop(columns=id_like_cols)
    test = test.drop(columns=[
        col for col in id_like_cols
        if col in test.columns
    ])

else:

    train_ids = pd.DataFrame(index=dataset.index)
    test_ids = pd.DataFrame(index=test.index)

print(
    f"Feature matrix after ID removal: {dataset.shape}"
)


Identifier-like columns detected: ['Id']
Feature matrix after ID removal: (1168, 80)


## Add missing-value indicator columns

For every column with missing values, add a companion `<col>_nan` flag (1 if missing, 0
otherwise) before filling — this preserves the "was this missing" signal, which can itself
be predictive (e.g. missing PoolQC often means "no pool", not "unknown").

In [5]:
features_with_nan = [f for f in dataset.columns if dataset[f].isnull().sum() > 0]

for feature in features_with_nan:
    dataset[feature + "_nan"] = np.where(dataset[feature].isnull(), 1, 0)

print(f"Added {len(features_with_nan)} missing-value indicator columns")
features_with_nan


Added 19 missing-value indicator columns


['LotFrontage',
 'Alley',
 'MasVnrType',
 'MasVnrArea',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Electrical',
 'FireplaceQu',
 'GarageType',
 'GarageYrBlt',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PoolQC',
 'Fence',
 'MiscFeature']

## Handle missing values — categorical features

In [6]:
features_nan_cat = [f for f in dataset.columns
                     if dataset[f].isnull().sum() > 0 and dataset[f].dtype == "O"]

for feature in features_nan_cat:
    pct = np.round(dataset[feature].isnull().mean(), 4)
    print(f"{feature} - {pct} missing")

def fill_categorical_nan(df, cols, fill_value="Missing"):
    df = df.copy()
    df[cols] = df[cols].fillna(fill_value)
    return df

dataset = fill_categorical_nan(dataset, features_nan_cat)
print("\nRemaining NaNs in these columns:", dataset[features_nan_cat].isnull().sum().sum())


Alley - 0.9366 missing
MasVnrType - 0.5848 missing
BsmtQual - 0.024 missing
BsmtCond - 0.024 missing
BsmtExposure - 0.024 missing
BsmtFinType1 - 0.024 missing
BsmtFinType2 - 0.024 missing
Electrical - 0.0009 missing
FireplaceQu - 0.4683 missing
GarageType - 0.0548 missing
GarageFinish - 0.0548 missing
GarageQual - 0.0548 missing
GarageCond - 0.0548 missing
PoolQC - 0.9949 missing
Fence - 0.8005 missing
MiscFeature - 0.9606 missing

Remaining NaNs in these columns: 0


## Handle missing values — numerical features

Medians are computed on train only and stored, so the exact same values fill test later (no leakage).

In [7]:
features_nan_num = [f for f in dataset.columns
                     if dataset[f].isnull().sum() > 0 and dataset[f].dtype != "O"]

for feature in features_nan_num:
    pct = np.round(dataset[feature].isnull().mean(), 4)
    print(f"{feature} - {pct} missing")

# Store medians computed on TRAIN ONLY, reused on test — never recompute medians on test data
train_medians = {f: dataset[f].median() for f in features_nan_num}

def fill_numeric_nan(df, medians):
    df = df.copy()
    for feature, median_value in medians.items():
        if feature in df.columns:
            df[feature] = df[feature].fillna(median_value)
    return df

dataset = fill_numeric_nan(dataset, train_medians)
print("\nRemaining NaNs in these columns:", dataset[features_nan_num].isnull().sum().sum())


LotFrontage - 0.1858 missing
MasVnrArea - 0.0051 missing
GarageYrBlt - 0.0548 missing

Remaining NaNs in these columns: 0


## Handle temporal columns (optional, dataset-specific)

Only runs if `TEMPORAL_COLS` and `TEMPORAL_REFERENCE_COL` are set in the config above —
converts e.g. `YearBuilt` into "years since built" relative to a reference column
(e.g. `YrSold`), which is usually more predictive than a raw year.

In [8]:
if TEMPORAL_COLS and TEMPORAL_REFERENCE_COL:
    for feature in TEMPORAL_COLS:
        dataset[feature] = dataset[TEMPORAL_REFERENCE_COL] - dataset[feature]
    print(f"Converted to relative years: {TEMPORAL_COLS}")
else:
    print("No temporal columns configured — skipping.")


No temporal columns configured — skipping.


## Handle skewed numeric features (auto-detected, not hardcoded)

Any numeric column (excluding the target and the nan-flag columns, which are already 0/1)
with `abs(skew) > SKEW_THRESHOLD` gets log-transformed. The target itself is also checked
and transformed separately, since it needs `np.expm1()` on predictions afterward.

In [9]:
from scipy import stats

nan_flag_cols = [f + "_nan" for f in features_with_nan]
numeric_cols = dataset.select_dtypes(include=[np.number]).columns.tolist()
candidate_cols = [c for c in numeric_cols if c not in nan_flag_cols and c != TARGET_COL]

skewed = dataset[candidate_cols].apply(lambda x: stats.skew(x.dropna()))
skewed_features = skewed[abs(skewed) > SKEW_THRESHOLD].index.tolist()

# Only log-transform strictly positive columns (log of 0/negative is invalid)
skewed_features = [c for c in skewed_features if (dataset[c] > 0).all()]

for feature in skewed_features:
    dataset[feature] = np.log(dataset[feature])

print(f"Log-transformed {len(skewed_features)} skewed columns: {skewed_features}")

# Transform the target too, if skewed
target_is_log_transformed = False
if TARGET_COL in dataset.columns and (dataset[TARGET_COL] > 0).all():
    target_skew = stats.skew(dataset[TARGET_COL])
    if abs(target_skew) > SKEW_THRESHOLD:
        dataset[TARGET_COL] = np.log(dataset[TARGET_COL])
        target_is_log_transformed = True
        print(f"Target \'{TARGET_COL}\' log-transformed (skew was {target_skew:.3f})")


Log-transformed 5 skewed columns: ['MSSubClass', 'LotFrontage', 'LotArea', '1stFlrSF', 'GrLivArea']
Target 'SalePrice' log-transformed (skew was 1.741)


## Handle rare categorical labels

Categories making up less than `RARE_LABEL_THRESHOLD` of rows get grouped into `"Rare_var"`.
This reduces overfitting to categories with very few examples. The kept-category list is
stored per column so test data uses the exact same grouping.

In [10]:
categorical_features = [f for f in dataset.columns if dataset[f].dtype == "O"]

frequent_labels = {}  # stored per column, reused on test

for feature in categorical_features:
    freq = dataset[feature].value_counts() / len(dataset)
    kept = freq[freq > RARE_LABEL_THRESHOLD].index
    frequent_labels[feature] = kept
    dataset[feature] = np.where(dataset[feature].isin(kept), dataset[feature], "Rare_var")

print(f"Applied rare-label grouping to {len(categorical_features)} categorical columns")


Applied rare-label grouping to 43 categorical columns


## Target-guided ordinal encoding

Each categorical column's labels are ranked by mean target value and mapped to integers —
this often works better than plain one-hot for tree/linear models on ordinal-like categories.
The mapping is stored per column so test data uses the identical encoding (any unseen
category on test falls back to NaN, then gets median-filled afterward).

In [11]:
label_mappings = {}  # stored per column, reused on test

for feature in categorical_features:
    labels_ordered = dataset.groupby(feature)[TARGET_COL].mean().sort_values().index
    mapping = {k: i for i, k in enumerate(labels_ordered, 0)}
    label_mappings[feature] = mapping
    dataset[feature] = dataset[feature].map(mapping)

print(f"Encoded {len(categorical_features)} categorical columns using target-guided ordinal mapping")
dataset.head()


Encoded 43 categorical columns using target-guided ordinal mapping


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,LotFrontage_nan,Alley_nan,MasVnrType_nan,MasVnrArea_nan,BsmtQual_nan,BsmtCond_nan,BsmtExposure_nan,BsmtFinType1_nan,BsmtFinType2_nan,Electrical_nan,FireplaceQu_nan,GarageType_nan,GarageYrBlt_nan,GarageFinish_nan,GarageQual_nan,GarageCond_nan,PoolQC_nan,Fence_nan,MiscFeature_nan
0,2.995732,3,4.248495,9.035987,1,2,0,1,1,1,0,7,2,1,3,4,5,6,1957,1957,0,0,4,2,1,0.0,1,2,2,2,3,1,1,922,4,0,392,1314,2,2,1,3,7.180831,0,0,7.180831,1,0,1,0,3,1,1,5,4,0,1,4,1957.0,2,1,294,3,3,2,250,0,0,0,0,0,0,3,2,0,6,2010,2,3,11.884489,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,1
1,4.094345,3,4.077537,8.966611,1,2,1,1,1,1,0,13,2,1,3,6,6,7,1993,1994,0,0,9,8,1,0.0,2,3,4,3,3,1,5,0,4,0,799,799,2,3,1,3,6.683361,772,0,7.359468,0,0,2,1,3,1,1,7,4,1,3,4,1993.0,2,2,380,3,3,2,0,40,0,0,0,0,0,3,2,0,5,2009,2,3,12.089539,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1
2,3.401197,3,4.204693,9.079890,1,2,0,1,1,1,0,3,1,1,3,4,5,7,1910,1950,0,0,4,1,1,0.0,1,3,2,1,3,1,5,0,4,0,796,796,2,3,1,2,6.679599,0,0,6.679599,0,0,1,0,2,1,1,4,4,0,1,0,1980.0,0,0,0,0,0,1,328,0,164,0,0,0,0,2,2,0,5,2008,2,3,11.350407,0,1,1,0,0,0,0,0,0,0,1,1,1,1,1,1,1,0,1
3,3.912023,3,4.094345,8.881836,1,2,0,1,1,0,0,6,1,1,3,1,5,7,1937,1950,0,0,3,1,2,252.0,1,3,1,3,3,1,4,569,4,0,162,731,2,4,1,3,6.888572,787,0,7.477604,1,0,1,1,3,1,2,7,4,2,3,2,1939.0,1,1,240,3,3,2,0,0,264,0,0,0,0,2,2,0,6,2007,2,3,12.072541,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1
4,3.912023,3,3.912023,8.517193,1,1,0,1,1,1,0,6,2,1,3,1,5,6,1924,1950,0,0,8,1,1,0.0,1,3,1,2,3,1,2,218,4,0,808,1026,2,2,1,3,6.933423,665,0,7.433075,0,0,2,0,3,1,2,6,4,1,4,2,1924.0,1,1,308,3,3,2,0,0,242,0,0,0,0,3,2,0,5,2010,2,3,11.751942,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1


## Feature scaling

MinMaxScaler is **fit on train only** and stored — the same fitted scaler transforms test
later (never re-fit on test data, which would leak test distribution info into the pipeline).

In [12]:
scalable_features = [f for f in dataset.columns if f != TARGET_COL]

scaler = MinMaxScaler()
scaler.fit(dataset[scalable_features])

dataset_scaled = pd.DataFrame(
    scaler.transform(dataset[scalable_features]),
    columns=scalable_features,
    index=dataset.index
)

# Reattach target (and ID, if present) for a complete, clean dataframe
data = pd.concat([dataset[[TARGET_COL]].reset_index(drop=True), dataset_scaled.reset_index(drop=True)], axis=1)
if ID_COL:
    data.insert(0, ID_COL, train_ids.reset_index(drop=True))

data.head()


,Id,SalePrice,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,LotFrontage_nan,Alley_nan,MasVnrType_nan,MasVnrArea_nan,BsmtQual_nan,BsmtCond_nan,BsmtExposure_nan,BsmtFinType1_nan,BsmtFinType2_nan,Electrical_nan,FireplaceQu_nan,GarageType_nan,GarageYrBlt_nan,GarageFinish_nan,GarageQual_nan,GarageCond_nan,PoolQC_nan,Fence_nan,MiscFeature_nan
0,255,11.884489,0.000000,0.75,0.445638,0.365182,1.0,1.0,0.000000,0.333333,1.0,0.25,0.0,0.333333,0.4,1.0,0.75,0.666667,0.444444,0.625,0.615942,0.116667,0.0,0.0,0.4,0.2,0.333333,0.000000,0.333333,0.666667,0.50,0.50,0.75,0.25,0.166667,0.163359,0.666667,0.0,0.167808,0.215057,1.0,0.50,1.0,1.000000,0.518336,0.000000,0.0,0.484528,0.333333,0.0,0.333333,0.0,0.375,0.333333,0.333333,0.250000,1.0,0.000000,0.2,0.8,0.518182,0.666667,0.25,0.207334,0.75,1.0,1.0,0.291715,0.000000,0.000000,0.0,0.0,0.0,0.0,0.75,1.0,0.0,0.454545,1.00,0.666667,0.75,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
1,1067,12.089539,0.487992,0.75,0.382360,0.351604,1.0,1.0,0.333333,0.333333,1.0,0.25,0.0,0.619048,0.4,1.0,0.75,1.000000,0.555556,0.750,0.876812,0.733333,0.0,0.0,0.9,0.8,0.333333,0.000000,0.666667,1.000000,1.00,0.75,0.75,0.25,0.833333,0.000000,0.666667,0.0,0.342038,0.130769,1.0,0.75,1.0,1.000000,0.330077,0.373850,0.0,0.547721,0.000000,0.0,0.666667,0.5,0.375,0.333333,0.333333,0.416667,1.0,0.333333,0.6,0.8,0.845455,0.666667,0.50,0.267983,0.75,1.0,1.0,0.000000,0.073126,0.000000,0.0,0.0,0.0,0.0,0.75,1.0,0.0,0.363636,0.75,0.666667,0.75,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
2,639,11.350407,0.180103,0.75,0.429425,0.373775,1.0,1.0,0.000000,0.333333,1.0,0.25,0.0,0.142857,0.2,1.0,0.75,0.666667,0.444444,0.750,0.275362,0.000000,0.0,0.0,0.4,0.1,0.333333,0.000000,0.333333,1.000000,0.50,0.25,0.75,0.25,0.833333,0.000000,0.666667,0.0,0.340753,0.130278,1.0,0.75,1.0,0.666667,0.328654,0.000000,0.0,0.307217,0.000000,0.0,0.333333,0.0,0.250,0.333333,0.333333,0.166667,1.0,0.000000,0.2,0.0,0.727273,0.000000,0.00,0.000000,0.00,0.0,0.5,0.382730,0.000000,0.297101,0.0,0.0,0.0,0.0,0.50,1.0,0.0,0.363636,0.50,0.666667,0.75,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
3,800,12.072541,0.407007,0.75,0.388581,0.335012,1.0,1.0,0.000000,0.333333,1.0,0.00,0.0,0.285714,0.2,1.0,0.75,0.166667,0.444444,0.750,0.471014,0.000000,0.0,0.0,0.3,0.1,0.666667,0.182874,0.333333,1.000000,0.25,0.75,0.75,0.25,0.666667,0.100815,0.666667,0.0,0.069349,0.119640,1.0,1.00,1.0,1.000000,0.407736,0.381114,0.0,0.589512,0.333333,0.0,0.333333,0.5,0.375,0.333333,0.666667,0.416667,1.0,0.666667,0.6,0.4,0.354545,0.333333,0.25,0.169252,0.75,1.0,1.0,0.000000,0.000000,0.478261,0.0,0.0,0.0,0.0,0.50,1.0,0.0,0.454545,0.25,0.666667,0.75,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,381,11.751942,0.407007,0.75,0.321097,0.263645,1.0,0.5,0.000000,0.333333,1.0,0.25,0.0,0.285714,0.4,1.0,0.75,0.166667,0.444444,0.625,0.376812,0.000000,0.0,0.0,0.8,0.1,0.333333,0.000000,0.333333,1.000000,0.25,0.50,0.75,0.25,0.333333,0.038625,0.666667,0.0,0.345890,0.167921,1.0,0.50,1.0,1.000000,0.424709,0.322034,0.0,0.573760,0.000000,0.0,0.666667,0.0,0.375,0.333333,0.666667,0.333333,1.0,0.333333,0.8,0.4,0.218182,0.333333,0

## Save engineered (pre-selection) training data

In [13]:
# The final user-facing outputs are X_train and X_test.
# No intermediate CSV is written here.
print(
    "Engineered training data prepared in memory — "
    f"shape {data.shape}"
)


Engineered training data prepared in memory — shape (1168, 100)


## Feature selection (Lasso + SelectFromModel)

Fits a Lasso regression and keeps only features with non-zero coefficients. `alpha` controls
strictness — higher alpha keeps fewer features. Works for regression targets; for a
classification target, swap `Lasso` for `LogisticRegression(penalty='l1', solver='liblinear')`.

In [14]:
feature_cols = [c for c in data.columns if c not in [TARGET_COL, ID_COL]]
X_train = data[feature_cols]
y_train = data[TARGET_COL]

feature_sel_model = SelectFromModel(Lasso(alpha=LASSO_ALPHA, random_state=0))
feature_sel_model.fit(X_train, y_train)

selected_feat = X_train.columns[feature_sel_model.get_support()]

print(f"Total features: {X_train.shape[1]}")
print(f"Selected features: {len(selected_feat)}")
print(f"Features shrunk to zero: {np.sum(feature_sel_model.estimator_.coef_ == 0)}")
print("\nSelected features:")
list(selected_feat)


Total features: 98
Selected features: 54
Features shrunk to zero: 44

Selected features:


['MSSubClass',
 'MSZoning',
 'LotArea',
 'LotShape',
 'LandContour',
 'LotConfig',
 'Neighborhood',
 'Condition1',
 'Condition2',
 'HouseStyle',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'YearRemodAdd',
 'RoofStyle',
 'RoofMatl',
 'Exterior1st',
 'Exterior2nd',
 'MasVnrType',
 'ExterQual',
 'ExterCond',
 'Foundation',
 'BsmtQual',
 'BsmtExposure',
 'BsmtFinType2',
 'BsmtUnfSF',
 'HeatingQC',
 'CentralAir',
 '1stFlrSF',
 '2ndFlrSF',
 'GrLivArea',
 'BsmtFullBath',
 'FullBath',
 'HalfBath',
 'KitchenAbvGr',
 'KitchenQual',
 'Functional',
 'Fireplaces',
 'FireplaceQu',
 'GarageFinish',
 'GarageCars',
 'GarageQual',
 'GarageCond',
 'PavedDrive',
 'WoodDeckSF',
 'ScreenPorch',
 'PoolQC',
 'Fence',
 'YrSold',
 'SaleType',
 'SaleCondition',
 'MasVnrType_nan',
 'PoolQC_nan',
 'Fence_nan']

In [15]:
X_train_selected = X_train[selected_feat]

final_train = pd.concat(
    [
        X_train_selected.reset_index(drop=True),
        y_train.reset_index(drop=True)
    ],
    axis=1
)

if ID_COL:
    final_train.insert(
        0,
        ID_COL,
        train_ids[ID_COL].reset_index(drop=True)
    )

final_train.to_csv(
    X_TRAIN_OUTPUT,
    index=False
)

print(
    f"Saved X_train: {X_TRAIN_OUTPUT}"
)
print(
    f"X_train shape: {final_train.shape}"
)

final_train.head()


Saved X_train: c:\Koustav\Programming\Machine Learning\Projects\Machine_learning_automation\Data_Preprocessing\outputs\X_train_20260811_193942.csv
X_train shape: (1168, 56)


,Id,MSSubClass,MSZoning,LotArea,LotShape,LandContour,LotConfig,Neighborhood,Condition1,Condition2,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,ExterQual,ExterCond,Foundation,BsmtQual,BsmtExposure,BsmtFinType2,BsmtUnfSF,HeatingQC,CentralAir,1stFlrSF,2ndFlrSF,GrLivArea,BsmtFullBath,FullBath,HalfBath,KitchenAbvGr,KitchenQual,Functional,Fireplaces,FireplaceQu,GarageFinish,GarageCars,GarageQual,GarageCond,PavedDrive,WoodDeckSF,ScreenPorch,PoolQC,Fence,YrSold,SaleType,SaleCondition,MasVnrType_nan,PoolQC_nan,Fence_nan,SalePrice
0,255,0.000000,0.75,0.365182,0.000000,0.333333,0.25,0.333333,0.4,1.0,0.666667,0.444444,0.625,0.615942,0.116667,0.0,0.0,0.4,0.2,0.333333,0.333333,0.666667,0.50,0.50,0.25,0.666667,0.167808,0.50,1.0,0.518336,0.000000,0.484528,0.333333,0.333333,0.0,0.333333,0.333333,1.0,0.000000,0.2,0.666667,0.25,0.75,1.0,1.0,0.291715,0.0,0.0,0.75,1.00,0.666667,0.75,1.0,1.0,1.0,11.884489
1,1067,0.487992,0.75,0.351604,0.333333,0.333333,0.25,0.619048,0.4,1.0,1.000000,0.555556,0.750,0.876812,0.733333,0.0,0.0,0.9,0.8,0.333333,0.666667,1.000000,1.00,0.75,0.25,0.666667,0.342038,0.75,1.0,0.330077,0.373850,0.547721,0.000000,0.666667,0.5,0.333333,0.333333,1.0,0.333333,0.6,0.666667,0.50,0.75,1.0,1.0,0.000000,0.0,0.0,0.75,0.75,0.666667,0.75,1.0,1.0,1.0,12.089539
2,639,0.180103,0.75,0.373775,0.000000,0.333333,0.25,0.142857,0.2,1.0,0.666667,0.444444,0.750,0.275362,0.000000,0.0,0.0,0.4,0.1,0.333333,0.333333,1.000000,0.50,0.25,0.25,0.666667,0.340753,0.75,1.0,0.328654,0.000000,0.307217,0.000000,0.333333,0.0,0.333333,0.333333,1.0,0.000000,0.2,0.000000,0.00,0.00,0.0,0.5,0.382730,0.0,0.0,0.50,0.50,0.666667,0.75,1.0,1.0,0.0,11.350407
3,800,0.407007,0.75,0.335012,0.000000,0.333333,0.00,0.285714,0.2,1.0,0.166667,0.444444,0.750,0.471014,0.000000,0.0,0.0,0.3,0.1,0.666667,0.333333,1.000000,0.25,0.75,0.25,0.666667,0.069349,1.00,1.0,0.407736,0.381114,0.589512,0.333333,0.333333,0.5,0.333333,0.666667,1.0,0.666667,0.6,0.333333,0.25,0.75,1.0,1.0,0.000000,0.0,0.0,0.50,0.25,0.666667,0.75,0.0,1.0,0.0,12.072541
4,381,0.407007,0.75,0.263645,0.000000,0.333333,0.25,0.285714,0.4,1.0,0.166667,0.444444,0.625,0.376812,0.000000,0.0,0.0,0.8,0.1,0.333333,0.333333,1.000000,0.25,0.50,0.25,0.666667,0.345890,0.50,1.0,0.424709,0.322034,0.573760,0.000000,0.666667,0.0,0.333333,0.666667,1.0,0.333333,0.8,0.333333,0.25,0.75,1.0,1.0,0.000000,0.0,0.0,0.75,1.00,0.666667,0.75,1.0,1.0,1.0,11.751942


## Apply the identical pipeline to test data

Uses only parameters already fit on train (medians, rare-label lists, encoding maps, scaler)
— nothing is re-fit here, which is what prevents train/test leakage.

In [16]:
# Apply the identical fitted preprocessing pipeline to X_test.
# The test set is never used to fit medians, encoders, scaler, or Lasso.

# The target is not included in X_test.
if TARGET_COL in test.columns:
    test_features = test.drop(columns=[TARGET_COL]).copy()
else:
    test_features = test.copy()

# Same missing-value indicator flags
for feature in features_with_nan:
    if feature in test_features.columns:
        test_features[feature + "_nan"] = np.where(
            test_features[feature].isnull(),
            1,
            0
        )

# Same categorical missing-value handling
test_features = fill_categorical_nan(
    test_features,
    [f for f in features_nan_cat if f in test_features.columns]
)

# Same numeric missing-value handling using TRAIN medians
test_features = fill_numeric_nan(
    test_features,
    train_medians
)

# Same temporal transform
if TEMPORAL_COLS and TEMPORAL_REFERENCE_COL:
    for feature in TEMPORAL_COLS:
        if feature in test_features.columns:
            test_features[feature] = (
                test_features[TEMPORAL_REFERENCE_COL]
                - test_features[feature]
            )

# Same skew log-transform
for feature in skewed_features:
    if feature in test_features.columns:
        # Train was only log-transformed when strictly positive.
        test_features[feature] = np.log(
            test_features[feature]
        )

# Same rare-label grouping using TRAIN-derived labels
for feature in categorical_features:
    if feature in test_features.columns:
        kept = frequent_labels[feature]

        test_features[feature] = np.where(
            test_features[feature].isin(kept),
            test_features[feature],
            "Rare_var"
        )

# Same target-guided encoding using TRAIN-derived mappings
for feature in categorical_features:
    if feature in test_features.columns:

        test_features[feature] = (
            test_features[feature]
            .map(label_mappings[feature])
        )

        # Unseen categories become NaN after mapping.
        if test_features[feature].isnull().sum() > 0:
            test_features[feature] = (
                test_features[feature]
                .fillna(dataset[feature].median())
            )

# Same scaler: transform only, never fit on test.
test_scaled = pd.DataFrame(
    scaler.transform(
        test_features[scalable_features]
    ),
    columns=scalable_features,
    index=test_features.index
)

# Keep only features selected from TRAIN.
test_final = test_scaled[selected_feat].copy()

# Preserve the ID if one was detected.
if ID_COL and ID_COL in test_ids.columns:
    test_final.insert(
        0,
        ID_COL,
        test_ids[ID_COL].reset_index(drop=True)
    )

test_final.to_csv(
    X_TEST_OUTPUT,
    index=False
)

print(
    f"Saved X_test: {X_TEST_OUTPUT}"
)
print(
    f"X_test shape: {test_final.shape}"
)

test_final.head()


Saved X_test: c:\Koustav\Programming\Machine Learning\Projects\Machine_learning_automation\Data_Preprocessing\outputs\X_test_20260811_193942.csv
X_test shape: (292, 55)


,Id,MSSubClass,MSZoning,LotArea,LotShape,LandContour,LotConfig,Neighborhood,Condition1,Condition2,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,ExterQual,ExterCond,Foundation,BsmtQual,BsmtExposure,BsmtFinType2,BsmtUnfSF,HeatingQC,CentralAir,1stFlrSF,2ndFlrSF,GrLivArea,BsmtFullBath,FullBath,HalfBath,KitchenAbvGr,KitchenQual,Functional,Fireplaces,FireplaceQu,GarageFinish,GarageCars,GarageQual,GarageCond,PavedDrive,WoodDeckSF,ScreenPorch,PoolQC,Fence,YrSold,SaleType,SaleCondition,MasVnrType_nan,PoolQC_nan,Fence_nan
0,893,0.000000,0.75,0.365508,0.000000,0.333333,0.25,0.238095,0.4,1.0,0.666667,0.555556,0.875,0.659420,0.883333,1.0,0.0,0.5,0.5,0.333333,0.333333,1.000000,0.5,0.5,0.25,0.666667,0.169521,0.50,1.0,0.439892,0.000000,0.411200,0.000000,0.333333,0.0,0.333333,0.333333,1.0,0.000000,0.2,0.666667,0.25,0.75,1.0,1.0,0.224037,0.0,0.0,0.50,0.00,0.666667,0.75,1.0,1.0,0.0
1,1106,0.487992,0.75,0.439121,0.333333,0.333333,0.00,1.000000,0.4,1.0,1.000000,0.777778,0.500,0.884058,0.750000,0.0,0.0,0.5,0.5,0.666667,0.666667,1.000000,1.0,1.0,0.75,0.666667,0.184503,1.00,1.0,0.568437,0.543341,0.728921,0.333333,0.666667,0.5,0.333333,0.666667,1.0,0.666667,0.6,0.666667,0.50,0.75,1.0,1.0,0.217036,0.0,0.0,0.75,1.00,0.666667,0.75,0.0,1.0,1.0
2,414,0.180103,0.25,0.377814,0.000000,0.333333,0.25,0.190476,0.0,1.0,0.666667,0.444444,0.625,0.398551,0.000000,0.0,0.0,0.2,0.3,0.333333,0.333333,1.000000,0.5,0.5,0.25,0.666667,0.431507,0.75,1.0,0.425446,0.000000,0.397696,0.000000,0.333333,0.0,0.333333,0.333333,1.0,0.333333,0.8,0.333333,0.50,0.75,1.0,1.0,0.000000,0.0,0.0,0.75,1.00,0.666667,0.75,1.0,1.0,1.0
3,523,0.407007,0.25,0.263645,0.000000,0.333333,0.00,0.095238,0.2,1.0,0.166667,0.555556,0.750,0.543478,0.000000,0.0,0.0,1.0,1.0,0.333333,0.333333,0.666667,0.5,0.5,0.25,0.666667,0.258990,1.00,1.0,0.416506,0.319613,0.568066,0.000000,0.666667,0.0,0.333333,0.333333,1.0,0.666667,0.8,0.333333,0.50,0.75,1.0,1.0,0.000000,0.0,0.0,0.75,0.00,0.666667,0.75,1.0,1.0,1.0
4,1037,0.000000,0.75,0.449114,0.333333,1.000000,0.25,0.857143,0.4,1.0,0.666667,0.888889,0.500,0.978261,0.966667,1.0,0.0,0.9,0.8,1.000000,0.666667,1.000000,1.0,1.0,1.00,0.666667,0.255993,1.00,1.0,0.597562,0.000000,0.558586,0.333333,0.666667,0.0,0.333333,1.000000,1.0,0.333333,1.0,1.000000,0.75,0.75,1.0,1.0,0.266044,0.0,0.0,0.75,0.75,0.666667,0.75,0.0,1.0,1.0


## Output files

Each run creates an `outputs` folder automatically if it does not already exist.

The final files are timestamped so previous runs are not overwritten:

- `X_train_YYYYMMDD_HHMMSS.csv`
- `X_test_YYYYMMDD_HHMMSS.csv`

The preprocessing parameters are fitted only on the training split and reused for
the test split.
